# 阶段 2：用 TRL 训练 Qwen2.5-0.5B 做 GRPO

> **目标**：用 TRL 库的 GRPOTrainer 训练 Qwen2.5-0.5B-Instruct 学会做两位数加法。
> 
> **你需要**：完成阶段 1（理解 GRPO 的 6 个步骤）
> **不需要**：任何 PyTorch 经验（TRL 帮你处理底层细节）

---

## 和阶段 1 的对应关系

你在阶段 1 用 numpy 手写了 GRPO 的每一步。现在 TRL 帮你做了这些，但**底层逻辑完全一样**：

| 阶段 1（numpy 手写） | 阶段 2（TRL 库） | 对应关系 |
|---------------------|-----------------|----------|
| `ToyPolicy` 类 | `Qwen2.5-0.5B-Instruct` | 策略模型 |
| `group_sampling()` | GRPOTrainer 内部自动做 | 组采样 |
| `compute_reward()` | 自定义 reward 函数传给 trainer | 奖励计算 |
| `compute_advantages()` | GRPOTrainer 内部自动做 | 优势标准化 |
| `policy_gradient_update()` | GRPOTrainer 内部自动做 | PPO 裁剪 + 梯度更新 |
| `apply_kl_penalty()` | `beta` 参数控制 | KL 惩罚 |
| `old_policy` 同步 | GRPOTrainer 内部自动做 | 旧策略同步 |

**一句话**：你不需要再手写梯度更新了，只需要告诉 TRL：
1. 用哪个模型
2. 用什么数据
3. 怎么计算奖励
4. 训练参数（G、lr、epsilon、beta 等）

---

## 学习路线

1. **Cell 1**：加载模型和 tokenizer
2. **Cell 2**：准备训练数据（两位数加法题）
3. **Cell 3**：定义奖励函数（和阶段 1 的 `compute_reward` 对应）
4. **Cell 4**：配置 GRPO 训练参数（和阶段 1 的超参数对应）
5. **Cell 5**：创建 GRPOTrainer 并开始训练
6. **Cell 6**：测试训练前后的模型对比
7. **Cell 7**：用 TensorBoard 可视化训练过程
8. **Cell 8**：理解检查点

## Cell 1：加载模型和 tokenizer

和阶段 1 的 `ToyPolicy()` 对应——这里换成真正的语言模型。

### Qwen2.5-0.5B-Instruct 是什么？

- 0.5B = 5 亿参数（非常小，适合在 16GB GPU 上训练）
- Instruct 版本 = 已经经过指令微调，能听懂人话
- 你在阶段 1 用的 `ToyPolicy` 只有几百个参数，这里有 5 亿个

### 为什么用 0.5B 而不是更大的模型？

- GRPO 训练需要为每个 prompt 生成 G 个回答（内存翻 G 倍）
- 0.5B 模型 + G=4 + max_completion_length=32 在 16GB GPU 上刚好够用
- 训练速度快，几分钟就能看到效果

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# ============================================================
# 加载模型和 tokenizer
# ============================================================

MODEL_PATH = "/data/models/Qwen2.5-0.5B-Instruct"

# Tokenizer：把文字变成 token ID（和阶段 1 的数字 0-9 对应）
# 阶段 1 的词表是 10 个数字，这里的词表是 15 万个 token
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
print(f"词表大小: {tokenizer.vocab_size}")
print(f"pad_token: {tokenizer.pad_token}")
print(f"eos_token: {tokenizer.eos_token}")

# 模型：5 亿参数的策略网络（和阶段 1 的 ToyPolicy 对应）
# 阶段 1 的模型是 W @ prompt + b + softmax
# 这里的模型是 Transformer + softmax，但输出的本质一样：每个 token 的概率分布
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.bfloat16,  # 用 bf16 节省显存（阶段 1 不需要因为模型太小）
    device_map="auto",          # 自动放到 GPU 上
)

print(f"\n模型参数量: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")
print(f"模型 dtype: {next(model.parameters()).dtype}")
print(f"模型设备: {next(model.parameters()).device}")

# 快速测试：让模型做一道加法题
# return_dict=False 让 apply_chat_template 返回纯 tensor 而不是 BatchEncoding
messages = [{"role": "user", "content": "What is 25+37? Answer with just the number."}]
input_ids = tokenizer.apply_chat_template(
    messages, return_tensors="pt", add_generation_prompt=True, return_dict=False
).to(model.device)
with torch.no_grad():
    output = model.generate(input_ids, max_new_tokens=20, temperature=1.0, do_sample=True)
response = tokenizer.decode(output[0][input_ids.shape[1]:], skip_special_tokens=True)
print(f"\n=== 训练前测试 ===")
print(f"问题: 25+37=?")
print(f"模型回答: {response.strip()}")
print(f"正确答案: 62")
print(f"（训练前模型可能答对也可能答错，我们来看看训练后能不能更稳定）")

## Cell 2：准备训练数据

和阶段 1 的 `generate_prompt()` 对应——这里换成真正的数学题。

### 数据格式

TRL 的 GRPOTrainer 要求数据集是一个 `Dataset` 对象，每条数据至少包含：
- `prompt`：给模型的问题（chat format）
- 奖励函数需要的额外字段（比如 `solution`：正确答案）

### 为什么用三位数加法？

- 两位数加法对 Qwen2.5-0.5B 太简单，组内经常全对，导致 `frac_reward_zero_std=1.0`，模型学不到东西
- 三位数加法更难，模型更容易答错，组内更容易出现"有对有错"的情况
- 有明确答案：方便设计奖励函数
- 数据无限：可以随机生成任意多道题

In [ ]:
import random
from datasets import Dataset

# ============================================================
# 生成三位数加法训练数据
# ============================================================

def generate_addition_dataset(n_samples=500, seed=42):
    """生成三位数加法数据集
    
    相比两位数加法，三位数加法更难，模型更容易答错，
    这样组内更容易出现"有对有错"的情况，提供更丰富的学习信号。
    
    每条数据包含：
    - prompt: chat format 的问题
    - solution: 正确答案（字符串形式，用于奖励函数比对）
    """
    random.seed(seed)
    data = []
    for _ in range(n_samples):
        a = random.randint(100, 999)
        b = random.randint(100, 999)
        answer = str(a + b)
        
        # TRL 要求 prompt 是 chat format（list of message dicts）
        prompt = [{
            "role": "user",
            "content": f"What is {a}+{b}? Answer with just the number."
        }]
        
        data.append({
            "prompt": prompt,
            "solution": answer,
        })
    
    return Dataset.from_list(data)

# 生成训练数据
train_dataset = generate_addition_dataset(n_samples=500)

print(f"训练数据量: {len(train_dataset)}")
print(f"\n=== 数据示例 ===")
for i in range(3):
    example = train_dataset[i]
    print(f"  prompt: {example['prompt'][0]['content']}")
    print(f"  solution: {example['solution']}")
    print()

## Cell 3：定义奖励函数

和阶段 1 的 `compute_reward()` 完全对应——检查模型回答是否正确。

### 阶段 1 vs 阶段 2 的奖励函数

| 阶段 1 | 阶段 2 |
|--------|--------|
| 检查是否降序排列 | 检查加法答案是否正确 |
| `r = 正确对数 / 4` | `r = 1.0（答对）或 0.0（答错）` |
| 输入：numpy 数组 | 输入：模型生成的文本 |

### TRL 奖励函数的接口

TRL 要求奖励函数的签名是：
```python
def reward_func(completions, **kwargs) -> list[float]
```

- `completions`：模型生成的 G 个回答（list of list of message dicts）
- `**kwargs`：数据集中的额外字段会自动传入（比如 `solution`）
- 返回值：每个回答的奖励值（list of float）

这和阶段 1 的 `compute_group_rewards(group, correct_answer)` 本质一样：
- 阶段 1：输入 numpy 数组，输出奖励列表
- 阶段 2：输入文本，输出奖励列表

In [ ]:
import re

# ============================================================
# 定义奖励函数
# ============================================================

def correctness_reward(completions, solution, **kwargs):
    """检查模型回答是否正确
    
    和阶段 1 的 compute_reward 对应：
    - 阶段 1：检查输出是否降序排列
    - 阶段 2：检查加法答案是否正确
    
    Args:
        completions: list of list of message dicts
            completions[i] = [{"role": "assistant", "content": "62"}]
        solution: list of str
            solution[i] = "62"
    
    Returns:
        list of float: 每个回答的奖励（1.0=正确, 0.0=错误）
    """
    rewards = []
    for completion, sol in zip(completions, solution):
        # 从 completion 中提取文本
        # completion 格式: [{"role": "assistant", "content": "some text"}]
        response_text = completion[0]["content"].strip()
        
        # 尝试从回答中提取数字
        # 模型可能回答 "62" 或 "The answer is 62" 等
        numbers = re.findall(r'\d+', response_text)
        
        if numbers and numbers[-1] == sol:
            rewards.append(1.0)  # 答对了
        else:
            rewards.append(0.0)  # 答错了
    
    return rewards

# 测试奖励函数
test_completions = [
    [{"role": "assistant", "content": "62"}],           # 正确
    [{"role": "assistant", "content": "The answer is 62"}],  # 正确（能提取到 62）
    [{"role": "assistant", "content": "63"}],           # 错误
    [{"role": "assistant", "content": "I don't know"}],  # 错误（没有数字）
]
test_solutions = ["62", "62", "62", "62"]

test_rewards = correctness_reward(test_completions, test_solutions)

print("=== 奖励函数测试 ===")
for i, (comp, sol, r) in enumerate(zip(test_completions, test_solutions, test_rewards)):
    print(f"  回答: '{comp[0]['content']}'  正确答案: {sol}  奖励: {r}")
print(f"\n奖励向量: {test_rewards}")
print(f"和阶段 1 一样：正确的回答得 1.0，错误的得 0.0")

## Cell 4：配置 GRPO 训练参数

和阶段 1 的 `train_grpo()` 函数参数对应——这里用 TRL 的 `GRPOConfig` 类。

### 参数对照表

| 阶段 1 参数 | 阶段 2 (GRPOConfig) | 含义 |
|------------|---------------------|------|
| `G=4` | `num_generations=6` | 每个 prompt 生成几个回答 |
| `lr=0.05` | `learning_rate=5e-6` | 学习率（LLM 用更小的 lr）|
| `epsilon=0.2` | `epsilon=0.2` | PPO 裁剪范围 |
| `beta=0.005` | `beta=0.1` | KL 惩罚权重 |
| `temperature=1.0` | `temperature=0.8` | 采样温度 |
| `num_steps=300` | `max_steps=300` | 训练步数 |

### 第二轮优化说明

第二轮训练（300步）暴露的核心问题：`frac_reward_zero_std` 几乎一直是 1.0——组内 4 个回答要么全对要么全错，模型学不到东西。

| 问题 | 上轮配置 | 本轮优化 | 原因 |
|------|---------|---------|------|
| 组内缺乏多样性 | G=4, temp=0.7 | G=6, temp=0.8 | 更大的组 + 略高的温度增加组内对错混合概率 |
| 题目太简单 | 两位数加法 | 三位数加法 | 更难的题让模型更容易答错，增加组内多样性 |
| 显存控制 | batch=4, G=4 (16序列) | batch=2, G=6 (12序列) | 序列数减少，显存反而更低 |

> **注意**：`generation_batch_size`（= `per_device_train_batch_size × gradient_accumulation_steps`）必须能被 `num_generations` 整除。2×3=6，6%6=0 ✓

In [ ]:
from trl import GRPOConfig

# ============================================================
# 配置 GRPO 训练参数（第二轮优化）
# ============================================================

training_args = GRPOConfig(
    output_dir="output",
    
    # === 组采样相关 ===
    num_generations=6,              # 4→6：更大的组，增加组内出现对错混合的概率
    max_completion_length=16,       # 三位数加法答案最多4位数，16 足够
    temperature=0.8,                # 0.7→0.8：略升温，增加组内多样性（0.7太确定，1.0太乱）
    
    # === 训练超参数 ===
    learning_rate=5e-6,             # 保持（配合 warmup 已经比较稳定）
    per_device_train_batch_size=2,  # 4→2：因为 G=6 增大了，降低 batch 省显存（2×6=12序列）
    gradient_accumulation_steps=3,  # 4→3：等效 batch_size=6（必须能被 G=6 整除）
    max_steps=300,                  # 保持 300 步
    warmup_steps=20,                # 保持 20 步 warmup
    
    # === PPO 裁剪 ===
    epsilon=0.2,                    # 裁剪范围 [0.8, 1.2]（不变）
    
    # === KL 惩罚 ===
    beta=0.1,                       # 保持 0.1
    
    # === 梯度控制 ===
    max_grad_norm=0.5,              # 保持梯度裁剪
    
    # === 精度和日志 ===
    bf16=True,                      # 用 bf16 训练（节省显存）
    logging_steps=1,                # 每步都记录日志
    save_steps=100,                 # 每 100 步保存一次检查点
    report_to=["tensorboard"],      # 日志写入 TensorBoard
    log_completions=True,           # 记录生成的回答（方便观察）
    num_completions_to_print=2,     # 每次打印 2 个回答示例
)

print("=== GRPO 训练配置（第二轮优化）===")
print(f"  num_generations (G): {training_args.num_generations}  (上轮: 4)")
print(f"  learning_rate: {training_args.learning_rate}")
print(f"  epsilon (clip): {training_args.epsilon}")
print(f"  beta (KL penalty): {training_args.beta}")
print(f"  max_steps: {training_args.max_steps}")
print(f"  temperature: {training_args.temperature}  (上轮: 0.7)")
print(f"  max_completion_length: {training_args.max_completion_length}")
print(f"  max_grad_norm: {training_args.max_grad_norm}")
print(f"  warmup_steps: {training_args.warmup_steps}")
print(f"  logging_steps: {training_args.logging_steps}")
print(f"  per_device_train_batch_size: {training_args.per_device_train_batch_size}  (上轮: 4)")
print(f"  gradient_accumulation_steps: {training_args.gradient_accumulation_steps}  (上轮: 2)")
print(f"  每步序列数: {training_args.per_device_train_batch_size * training_args.num_generations}  (上轮: 16)")
print(f"  等效 batch_size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"  bf16: {training_args.bf16}")
print(f"\n-> 优化思路：G=6增组内多样性、温度0.8平衡确定性与探索、三位数加法增加难度")

## Cell 5：创建 GRPOTrainer 并开始训练

这一步把所有东西串起来，和阶段 1 的 `train_grpo()` 函数对应。

### TRL 帮你做了什么？

你在阶段 1 手写的这些步骤，TRL 全部内置了：
- 组采样（`group_sampling`）-> GRPOTrainer 自动做
- 优势计算（`compute_advantages`）-> GRPOTrainer 自动做
- 策略更新（`policy_gradient_update`）-> GRPOTrainer 自动做
- KL 惩罚（`apply_kl_penalty`）-> GRPOTrainer 自动做
- 旧策略同步（`old_policy = deepcopy(policy)`）-> GRPOTrainer 自动做

你只需要提供：模型、数据、奖励函数、配置参数。

### 训练时会发生什么？

```
每一步训练：
  1. 取一批 prompt（加法题）
  2. 每个 prompt 生成 G=4 个回答（组采样）
  3. 用 correctness_reward 给每个回答打分
  4. 组内标准化得到优势
  5. PPO 裁剪 + 梯度更新
  6. KL 惩罚
  -> 和阶段 1 的流程一模一样！
```

### 预期时间

在 RTX 5070 Ti 上，50 步大约需要 5-10 分钟。

In [ ]:
from trl import GRPOTrainer

# ============================================================
# 创建 GRPOTrainer
# ============================================================

trainer = GRPOTrainer(
    model=MODEL_PATH,                # 模型路径（TRL 会自动加载）
    reward_funcs=correctness_reward,  # 奖励函数（和阶段 1 的 compute_reward 对应）
    args=training_args,               # 训练配置
    train_dataset=train_dataset,      # 训练数据
)

print("=== GRPOTrainer 创建完成 ===")
print(f"模型: {MODEL_PATH}")
print(f"奖励函数: correctness_reward")
print(f"训练数据: {len(train_dataset)} 道加法题")
print(f"\n开始训练...")

# ============================================================
# 开始训练！
# ============================================================
# 这一行等价于阶段 1 的整个 train_grpo() 循环
# TRL 内部自动完成了组采样、奖励计算、优势标准化、PPO更新、KL惩罚

trainer.train()

# 保存训练后的模型
trainer.save_model("output/grpo_addition_model")
print(f"\n=== 训练完成 ===")
print(f"模型已保存到 output/grpo_addition_model")

## Cell 6：训练前后对比

和阶段 1 Cell 9 的训练前后对比对应——用相同的题目测试训练前后的模型。

### 你应该看到什么？

- 训练前：基座模型在三位数加法上的准确率（作为基线）
- 训练后：经过 GRPO 训练后的准确率变化

注意：300 步训练只是演示，效果取决于训练质量。如果 `frac_reward_zero_std` 较高（组内缺乏多样性），学习信号不足，效果可能不明显。

In [ ]:
import random

# ============================================================
# 训练前后对比测试（三位数加法）
# ============================================================

# 生成 20 道三位数测试题（和训练数据同难度，但用不同 seed 确保不重复）
random.seed(999)
test_questions = []
for _ in range(20):
    a = random.randint(100, 999)
    b = random.randint(100, 999)
    test_questions.append((a, b, str(a + b)))

def test_model(model, tokenizer, questions, label="Model"):
    """测试模型在加法题上的准确率"""
    correct = 0
    results = []
    for a, b, ans in questions:
        messages = [{"role": "user", "content": f"What is {a}+{b}? Answer with just the number."}]
        input_ids = tokenizer.apply_chat_template(
            messages, return_tensors="pt", add_generation_prompt=True, return_dict=False
        ).to(model.device)
        
        with torch.no_grad():
            output = model.generate(
                input_ids, max_new_tokens=16, temperature=0.0, do_sample=False
            )
        
        response = tokenizer.decode(output[0][input_ids.shape[1]:], skip_special_tokens=True).strip()
        numbers = re.findall(r'\d+', response)
        is_correct = bool(numbers and numbers[-1] == ans)
        
        if is_correct:
            correct += 1
        results.append((f"{a}+{b}", ans, response, is_correct))
    
    accuracy = correct / len(questions)
    return accuracy, results

# 测试训练前的模型
print("=== 测试训练前模型 ===")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, torch_dtype=torch.bfloat16, device_map="auto"
)
base_acc, base_results = test_model(base_model, tokenizer, test_questions, "Before")
print(f"准确率: {base_acc:.1%} ({sum(r[3] for r in base_results)}/{len(base_results)})")

# 测试训练后的模型
print(f"\n=== 测试训练后模型 ===")
trained_model = AutoModelForCausalLM.from_pretrained(
    "output/grpo_addition_model", torch_dtype=torch.bfloat16, device_map="auto"
)
trained_acc, trained_results = test_model(trained_model, tokenizer, test_questions, "After")
print(f"准确率: {trained_acc:.1%} ({sum(r[3] for r in trained_results)}/{len(trained_results)})")

# 打印对比
print(f"\n=== 对比 ===")
print(f"训练前准确率: {base_acc:.1%}")
print(f"训练后准确率: {trained_acc:.1%}")
print(f"提升: {trained_acc - base_acc:+.1%}")
print(f"\n（300 步训练，三位数加法，G=6，temp=0.8）")

# 打印几个具体例子
print(f"\n=== 具体例子 ===")
print(f"{'题目':>12} | {'正确答案':>6} | {'训练前':>14} | {'训练后':>14}")
print("-" * 58)
for i in range(min(10, len(base_results))):
    q, ans, before_resp, before_ok = base_results[i]
    _, _, after_resp, after_ok = trained_results[i]
    print(f"{q:>12} | {ans:>6} | {before_resp[:14]:>14} | {after_resp[:14]:>14}")

# 释放显存
del base_model, trained_model
torch.cuda.empty_cache()

=== 测试训练前模型 ===


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 4496.22it/s]


准确率: 50.0% (10/20)

=== 测试训练后模型 ===


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 2068.50it/s]


准确率: 80.0% (16/20)

=== 对比 ===
训练前准确率: 50.0%
训练后准确率: 80.0%
提升: +30.0%

（300 步训练，三位数加法，G=6，temp=0.8）

=== 具体例子 ===
          题目 |   正确答案 |            训练前 |            训练后
----------------------------------------------------------
     900+795 |   1695 |           1795 |           1695
     181+993 |   1174 |           2104 | 181 + 993 = 11
     681+687 |   1368 |           1374 |           1368
     646+602 |   1248 |           1258 |           1248
     594+235 |    829 |            829 |           1849
     999+917 |   1916 |           1816 |           1816
     762+425 |   1187 |           1187 |           1187
     759+913 |   1672 |           1672 |           1672
     758+965 |   1723 |           1723 |           1723
     938+199 |   1137 |           1137 |           1137


: 

## Cell 7：用 TensorBoard 可视化

和阶段 1 Cell 9 的 matplotlib 可视化对应——这里用 TensorBoard 看训练曲线。

### 关键指标（和阶段 1 一一对应）

| TensorBoard 指标 | 阶段 1 对应 | 含义 |
|-----------------|-------------|------|
| `reward` | `reward_mean` | 平均奖励（应该上升）|
| `reward_std` | `reward_std` | 奖励标准差（探索性）|
| `kl` | `kl` | KL 散度（应保持合理）|
| `loss` | — | 总损失 |

### 启动 TensorBoard

在终端运行：
```bash
conda activate grpo-tutorial
tensorboard --logdir output --port 6006
```
然后在浏览器打开 http://localhost:6006

In [ ]:
# ============================================================
# 查看训练日志
# ============================================================

import os

# 查找 TensorBoard 日志目录
log_dir = "output"
if os.path.exists(log_dir):
    print(f"TensorBoard 日志目录: {log_dir}")
    print(f"\n在终端运行以下命令启动 TensorBoard：")
    print(f"  conda activate grpo-tutorial")
    print(f"  tensorboard --logdir {log_dir} --port 6006")
    print(f"\n然后在浏览器打开: http://localhost:6006")
else:
    print("未找到日志目录，请确认训练已完成")

# 打印训练过程中记录的关键指标
print(f"\n=== 训练日志摘要 ===")
print(f"关键指标（和阶段 1 对应）：")
print(f"  reward     -> 平均奖励（对应阶段1的 reward_mean，应该上升）")
print(f"  reward_std -> 奖励标准差（对应阶段1的 reward_std，反映探索性）")
print(f"  kl         -> KL散度（对应阶段1的 kl，应保持合理范围）")
print(f"  loss       -> 总损失")
print(f"\n-> 和阶段 1 的 matplotlib 图形本质完全一样，只是换成了 TensorBoard")

## Cell 8：理解检查点

### 检查你的理解

1. **TRL 的 GRPOTrainer 帮你做了阶段 1 中的哪些步骤？**
   - 提示：组采样、优势计算、策略更新、KL 惩罚、旧策略同步

2. **你还需要自己提供什么？**
   - 提示：模型、数据、奖励函数、配置参数

3. **`num_generations=4` 对应阶段 1 的哪个参数？**
   - 提示：G

4. **为什么学习率从 0.05 降到 1e-5？**
   - 提示：模型参数从几百个变成 5 亿个

5. **奖励函数的接口和阶段 1 有什么区别？**
   - 提示：输入从 numpy 数组变成文本，但输出都是奖励列表

6. **`beta=0.04` 对应阶段 1 的什么？如果设为 0 会怎样？**
   - 提示：KL 惩罚权重，设为 0 则不限制模型偏离

### 动手实验建议

```python
# 实验 1：增加训练步数
# 把 max_steps 从 50 改成 200，观察准确率是否更高

# 实验 2：增大组大小
# 把 num_generations 从 4 改成 8（注意显存！）

# 实验 3：去掉 KL 惩罚
# 把 beta 设为 0，观察 KL 散度是否爆炸

# 实验 4：换更难的题目
# 把两位数加法换成三位数加法或乘法
```

### 下一步

如果你能回答以上所有问题，恭喜！你已经会用 TRL 做 GRPO 训练了。

接下来在**阶段 3**中，我们会阅读 TRL 的 GRPOTrainer 源码，看看 TRL 内部是怎么实现组采样、优势计算、PPO 裁剪的——你会发现它们和你在阶段 1 手写的代码几乎一模一样。